# NB-05: 想定めぐ指数の集計・検証

NB-04で算出した実測めぐ指数を集計し、出走予定レースに対する各馬の「期待パフォーマンス」を表す「想定めぐ指数」を計算する。

## 集計パターン
- **パターンA**: 直近5走の最大値
- **パターンB**: 直近5走の加重平均 (0.35/0.25/0.20/0.12/0.08)
- **パターンC**: 直近3走の最大値（距離帯・surface 一致条件）

## 処理フロー
1. セットアップ・データロード
2. タイムライン整合性確保（テンポラルリーク防止）
3. パターンA/B/C 計算
4. 検証
5. 保存

## セクション1: セットアップ・データロード

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ---- 入出力パス定義 ----
NB04_DIR = Path('/home/jovyan/work/keiba-vpn/notebooks/megu_index/output/nb04')
OUTPUT_DIR = Path('/home/jovyan/work/keiba-vpn/notebooks/megu_index/output/nb05')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('パス設定完了')

In [ ]:
# ---- データ読み込み ----
print('megu_index.parquet 読み込み中...')
df = pd.read_parquet(NB04_DIR / 'megu_index.parquet')
print(f'  megu_index: {len(df):,} 行')
print(f'  カラム: {df.columns.tolist()}')
print(f'  date 範囲: {df["date"].min()} 〜 {df["date"].max()}')
print(f'  horse_id ユニーク数: {df["horse_id"].nunique():,}')
print(f'  race_id ユニーク数: {df["race_id"].nunique():,}')
print('データ読み込み完了')

## セクション2: タイムライン整合性確保

In [ ]:
print('タイムライン整合性処理中...')

# date を datetime 型に変換
df['date'] = pd.to_datetime(df['date'])

# horse_id × date 昇順でソート（テンポラルリーク防止のため）
df = df.sort_values(['horse_id', 'date', 'race_id']).reset_index(drop=True)

# valid な走のみ集計対象（megu_index が null でないもの）
df_valid = df[df['megu_index'].notna()].copy()
print(f'valid レコード数: {len(df_valid):,} / {len(df):,}')

# 距離帯の定義（パターンC で使用）
def get_distance_band(dist: int) -> str:
    """距離をバンドに変換する。"""
    if dist <= 1400:
        return 'sprint'
    elif dist <= 1800:
        return 'mile'
    elif dist <= 2200:
        return 'intermediate'
    else:
        return 'long'

df['distance_band'] = df['distance'].apply(get_distance_band)
df_valid['distance_band'] = df_valid['distance'].apply(get_distance_band)
print('距離帯付与完了')
print(df['distance_band'].value_counts())

In [ ]:
# ---- 各レースエントリーに対して「そのレース以前の走」を取得するための準備 ----
print('レースエントリーリスト作成中...')

# 全レースエントリー（レース × 馬のすべての組み合わせ）
entries = df[['race_id', 'horse_id', 'date', 'surface', 'distance', 'distance_band']].drop_duplicates()
entries = entries.sort_values(['horse_id', 'date', 'race_id']).reset_index(drop=True)
print(f'エントリー数: {len(entries):,}')

## セクション3: パターンA/B/C 計算

各馬について、各レース出走時点での「過去走の megu_index」を取得し、
パターンA/B/C でそれぞれ集計する。

実装方針:
- groupby + shift でラグ特徴量を作成
- cumcount を使って各馬の走順（rank）を付与
- テンポラルリーク防止のため、当該レースの前走のみを使用

In [ ]:
# ---- 各馬の過去走ラグ特徴量を生成（全レース対象） ----
print('ラグ特徴量生成中...')

N_LAGS = 5
WEIGHTS_B = {1: 0.35, 2: 0.25, 3: 0.20, 4: 0.12, 5: 0.08}

# horse_id × date 昇順でソート済み
# shift(1) = 直前の走、shift(2) = 2走前、...
df_sorted = df.sort_values(['horse_id', 'date', 'race_id']).copy()

for lag in range(1, N_LAGS + 1):
    df_sorted[f'prev_megu_{lag}'] = (
        df_sorted.groupby('horse_id')['megu_index']
        .shift(lag)
    )
    # パターンC 用: 過去走の surface と distance_band も付与
    df_sorted[f'prev_surface_{lag}'] = (
        df_sorted.groupby('horse_id')['surface']
        .shift(lag)
    )
    df_sorted[f'prev_dist_band_{lag}'] = (
        df_sorted.groupby('horse_id')['distance_band']
        .shift(lag)
    )

print('ラグ特徴量生成完了')
print(f'カラム数: {len(df_sorted.columns)}')

In [ ]:
# ---- パターンA: 直近5走の最大値 ----
print('パターンA 計算中（直近5走の最大値）...')

prev_cols = [f'prev_megu_{i}' for i in range(1, N_LAGS + 1)]
df_sorted['megu_a'] = df_sorted[prev_cols].max(axis=1)

print(f'megu_a 統計:')
print(df_sorted['megu_a'].describe())
print(f'megu_a null 率: {df_sorted["megu_a"].isna().mean():.1%}')

In [ ]:
# ---- パターンB: 直近5走の加重平均 ----
print('パターンB 計算中（直近5走の加重平均）...')

def weighted_avg_b(row):
    """直近5走の加重平均を計算（null を除く）。"""
    total_weight = 0.0
    total_val = 0.0
    for lag, w in WEIGHTS_B.items():
        val = row.get(f'prev_megu_{lag}', np.nan)
        if pd.notna(val):
            total_val += w * val
            total_weight += w
    if total_weight == 0.0:
        return np.nan
    return total_val / total_weight

# ベクトル化実装（apply より高速）
numerator = sum(
    WEIGHTS_B[lag] * df_sorted[f'prev_megu_{lag}'].fillna(0)
    * df_sorted[f'prev_megu_{lag}'].notna().astype(float)
    for lag in range(1, N_LAGS + 1)
)
denominator = sum(
    WEIGHTS_B[lag] * df_sorted[f'prev_megu_{lag}'].notna().astype(float)
    for lag in range(1, N_LAGS + 1)
)
df_sorted['megu_b'] = np.where(denominator > 0, numerator / denominator, np.nan)

print(f'megu_b 統計:')
print(df_sorted['megu_b'].describe())
print(f'megu_b null 率: {df_sorted["megu_b"].isna().mean():.1%}')

In [ ]:
# ---- パターンC: 同 distance_band × surface の直近3走の最大値 ----
print('パターンC 計算中（同distance_band × surface の直近3走の最大値）...')

def calc_megu_c(row):
    """同距離帯・同surface の直近3走の最大値を返す。"""
    cur_surface = row['surface']
    cur_dist_band = row['distance_band']
    vals = []
    for lag in range(1, 6):  # 過去5走以内から同条件を探す
        if len(vals) >= 3:
            break
        prev_surf = row.get(f'prev_surface_{lag}')
        prev_db = row.get(f'prev_dist_band_{lag}')
        prev_val = row.get(f'prev_megu_{lag}')
        if (
            pd.notna(prev_surf)
            and prev_surf == cur_surface
            and pd.notna(prev_db)
            and prev_db == cur_dist_band
            and pd.notna(prev_val)
        ):
            vals.append(prev_val)
    return max(vals) if vals else np.nan

df_sorted['megu_c'] = df_sorted.apply(calc_megu_c, axis=1)

print(f'megu_c 統計:')
print(df_sorted['megu_c'].describe())
print(f'megu_c null 率: {df_sorted["megu_c"].isna().mean():.1%}')

In [ ]:
# ---- n_valid_runs 計算（各レース時点での過去valid走数）----
print('n_valid_runs 計算中...')

df_sorted['n_valid_runs'] = (
    df_sorted[prev_cols].notna().sum(axis=1).astype(int)
)

print(f'n_valid_runs 分布:')
print(df_sorted['n_valid_runs'].value_counts().sort_index())

## セクション4: 検証

In [ ]:
# ---- 4-1: A/B/C の相関行列 ----
print('=== パターンA/B/C 相関行列 ===')

corr_df = df_sorted[['megu_a', 'megu_b', 'megu_c']].dropna()
print(f'有効行数: {len(corr_df):,}')
print(corr_df.corr().round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
pairs = [('megu_a', 'megu_b'), ('megu_a', 'megu_c'), ('megu_b', 'megu_c')]
for ax, (x_col, y_col) in zip(axes, pairs):
    subset = df_sorted[[x_col, y_col]].dropna()
    ax.scatter(subset[x_col], subset[y_col], alpha=0.1, s=5)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    corr = subset[x_col].corr(subset[y_col])
    ax.set_title(f'{x_col} vs {y_col}\n(r={corr:.3f})')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'megu_abc_correlation.png', dpi=100)
plt.show()
print('相関行列プロット 保存完了')

In [ ]:
# ---- 4-2: null 率（過去走なし・新馬等）----
print('=== null 率の確認 ===')
null_rates = {
    'megu_a': df_sorted['megu_a'].isna().mean(),
    'megu_b': df_sorted['megu_b'].isna().mean(),
    'megu_c': df_sorted['megu_c'].isna().mean(),
}
for k, v in null_rates.items():
    print(f'  {k}: {v:.1%}')

# n_valid_runs=0（新馬・デビュー戦）の確認
debut_rate = (df_sorted['n_valid_runs'] == 0).mean()
print(f'\n  過去走なし（デビュー戦等）: {debut_rate:.1%}')

In [ ]:
# ---- 4-3: 分布の確認 ----
print('=== A/B/C 分布の確認 ===')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800']

for ax, col, color in zip(axes, ['megu_a', 'megu_b', 'megu_c'], colors):
    data = df_sorted[col].dropna()
    ax.hist(data, bins=50, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(50, color='red', linestyle='--', label='基準値=50')
    ax.set_title(f'{col} 分布\n平均={data.mean():.1f}, std={data.std():.1f}')
    ax.set_xlabel('想定めぐ指数')
    ax.set_ylabel('頻度')
    ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'megu_abc_distribution.png', dpi=100)
plt.show()
print('分布プロット 保存完了')

## セクション5: 保存

In [ ]:
# ---- 出力スキーマに合わせてカラムを選択 ----
output_cols = [
    'race_id', 'horse_id', 'date',
    'megu_a', 'megu_b', 'megu_c',
    'n_valid_runs'
]

available_cols = [c for c in output_cols if c in df_sorted.columns]
missing_cols = [c for c in output_cols if c not in df_sorted.columns]
if missing_cols:
    print(f'警告: 以下のカラムが存在しません: {missing_cols}')

df_out = df_sorted[available_cols].copy()

# 型変換
df_out['race_id'] = df_out['race_id'].astype(str)
df_out['horse_id'] = df_out['horse_id'].astype(str)
df_out['date'] = df_out['date'].astype(str)
df_out['n_valid_runs'] = df_out['n_valid_runs'].astype(int)

print(f'出力データ: {len(df_out):,} 行 × {len(df_out.columns)} 列')
print(df_out.dtypes)

In [ ]:
# ---- megu_final.parquet 保存 ----
output_path = OUTPUT_DIR / 'megu_final.parquet'
df_out.to_parquet(output_path, index=False)
print(f'megu_final.parquet 保存完了: {output_path}')
print(f'  行数: {len(df_out):,}')
for col in ['megu_a', 'megu_b', 'megu_c']:
    print(f'  {col} null 率: {df_out[col].isna().mean():.1%}')

# 読み込み検証
verify = pd.read_parquet(output_path)
print(f'\n検証読み込み OK: {len(verify):,} 行')
print('NB-05 処理完了')